# 01 · Forward model — worked solutions

These solutions are most useful after attempting the chapter exercises. Each
section states the reasoning before the calculation; numerical values depend on
the compact, fixed-seed setup below.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import geodef

rng = np.random.default_rng(0)
fault = geodef.Fault.planar(
    lat=34.0, lon=-118.0, depth=9_000.0, strike=90.0, dip=30.0,
    length=24_000.0, width=12_000.0, n_length=4, n_width=3,
)
i = np.arange(fault.n_patches) % 4
j = np.arange(fault.n_patches) // 4
true_slip = np.exp(-((i - 1.5) / 1.2) ** 2 - ((j - 1.0) / 0.9) ** 2)
station_lon, station_lat = np.meshgrid(
    np.linspace(-118.18, -117.82, 6), np.linspace(33.90, 34.12, 5)
)
station_lon, station_lat = station_lon.ravel(), station_lat.ravel()
east, north, up = fault.displacement(
    station_lat, station_lon, slip_strike=0.0, slip_dip=true_slip
)
sigma = 0.004
gnss = geodef.data.gnss(
    lon=station_lon, lat=station_lat,
    east=east + rng.normal(0.0, sigma, east.size),
    north=north + rng.normal(0.0, sigma, north.size),
    up=up + rng.normal(0.0, sigma, up.size),
    sigma_east=sigma, sigma_north=sigma, sigma_up=sigma,
    name="solution_gnss",
)


## 1–2. Dip, depth, and mechanism

Shallower sources have larger, sharper surface gradients. Steeper dip rotates
the balance of horizontal and vertical motion. Pure strike slip changes the
dip-slip lobe into a four-quadrant horizontal pattern. Compare peak norm with
one shared station grid.

## 3. Moment magnitude

Compute area times slip before the logarithm. Doubling area or slip doubles
moment but adds only about 0.2 magnitude units.

## 4. Superposition

The two-source prediction must equal the prediction from summed physical
components to floating-point tolerance.


In [ ]:
peaks = {}
for test_dip in (15.0, 60.0):
    test_fault = geodef.Fault.planar(
        lat=34.0, lon=-118.0, depth=9_000.0, strike=90.0, dip=test_dip,
        length=24_000.0, width=12_000.0, n_length=4, n_width=3,
    )
    de, dn, du = test_fault.displacement(
        station_lat, station_lon, slip_strike=0.0, slip_dip=true_slip
    )
    peaks[test_dip] = np.max(np.sqrt(de**2 + dn**2 + du**2))

mw = fault.magnitude(true_slip)
first = fault.displacement(station_lat, station_lon, 0.0, true_slip * 0.4)
second = fault.displacement(station_lat, station_lon, 0.0, true_slip * 0.6)
total = fault.displacement(station_lat, station_lon, 0.0, true_slip)
assert all(np.allclose(a + b, c) for a, b, c in zip(first, second, total, strict=True))
print("peak displacement by dip:", peaks, "Mw:", mw)


## Interpretation and alternatives

Linearity makes superposition exact only because geometry and medium stay fixed.
